In [ ]:
from google.colab import files
uploaded = files.upload()

Saving netflix_recommendation_dataset.csv to netflix_recommendation_dataset.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
PALETTE   = ['#E50914','#221F1F','#B81D24','#F5F5F1','#6B6B6B']
BG        = '#141414'
TEXT      = '#F5F5F1'
ACCENT    = '#E50914'
CARD      = '#1F1F1F'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': '#333', 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': '#2a2a2a',
    'grid.linewidth': 0.5, 'font.family': 'DejaVu Sans',
})

df = pd.read_csv('netflix_recommendation_dataset.csv')
print(f"Dataset: {df.shape[0]} rows | {df['customer_id'].nunique()} users | {df['movie_id'].nunique()} movies")


Dataset: 17770 rows | 1444 users | 100 movies


In [ ]:
df = pd.read_csv("netflix_recommendation_dataset.csv")

df.head()

,customer_id,movie_id,movie_name,genre,rating
0,U0001,M004,Interstellar,Sci-Fi,5
1,U0001,M095,The Martian,Sci-Fi,1
2,U0001,M036,Se7en,Thriller,5
3,U0001,M032,The Pursuit of Happyness,Drama,5
4,U0001,M029,Blade Runner 2049,Sci-Fi,1


OBJECTIVE 1 – Most Popular & Liked Genres

In [ ]:
popular_genre = df.groupby("genre")["rating"].count().sort_values(ascending=False)

print("Most Popular Genres:")
print(popular_genre)

Most Popular Genres:
genre
Sci-Fi         2708
Action         2535
Horror         2279
Comedy         1981
Drama          1953
Thriller       1783
Romance        1699
Animation      1093
Fantasy         908
Documentary     831
Name: rating, dtype: int64


In [ ]:
liked_genre = df.groupby("genre")["rating"].mean().sort_values(ascending=False)

print("Most Liked Genres:")
print(liked_genre)

Most Liked Genres:
genre
Animation      4.235133
Drama          3.839222
Thriller       3.789119
Fantasy        3.603524
Documentary    3.049338
Sci-Fi         3.037666
Comedy         3.021706
Romance        3.015892
Action         3.008284
Horror         2.985081
Name: rating, dtype: float64


In [ ]:
user_movie_matrix = df.pivot_table(
    index='customer_id',
    columns='movie_name',
    values='rating'
).fillna(0)

user_movie_matrix.head()

movie_name,12 Angry Men,13th,2001: A Space Odyssey,A Beautiful Mind,A Quiet Place,Airplane!,Alien,Anchorman,Annihilation,Arrival,...,Thor: Ragnarok,Titanic,Top Gun: Maverick,Toy Story,Up,Us,WALL-E,When Harry Met Sally,Whiplash,Zodiac
customer_id,,,,,,,,,,,,,,,,,,,,,
U0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0003,0.0,1.0,2.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
U0004,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0005,0.0,0.0,5.0,0.0,0.0,0.0,4.0,4.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0


Apply SVD (Recommendation Model)

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=20, random_state=42)

matrix = user_movie_matrix.values

svd_matrix = svd.fit_transform(matrix)

reconstructed_matrix = np.dot(svd_matrix, svd.components_)

predicted_ratings = pd.DataFrame(
    reconstructed_matrix,
    columns=user_movie_matrix.columns,
    index=user_movie_matrix.index
)

predicted_ratings.head()

movie_name,12 Angry Men,13th,2001: A Space Odyssey,A Beautiful Mind,A Quiet Place,Airplane!,Alien,Anchorman,Annihilation,Arrival,...,Thor: Ragnarok,Titanic,Top Gun: Maverick,Toy Story,Up,Us,WALL-E,When Harry Met Sally,Whiplash,Zodiac
customer_id,,,,,,,,,,,,,,,,,,,,,
U0001,0.680569,-0.297498,-0.143072,-1.184764,0.192255,0.411807,0.262721,0.049014,0.086350,0.188237,...,-0.390246,0.728812,0.647530,-0.146023,-0.194239,0.352838,0.426741,0.144155,-0.267427,0.204764
U0002,-0.267580,0.136617,0.402051,0.239269,0.669466,0.311904,0.276109,0.145302,0.478146,0.273433,...,0.308364,0.306902,0.506511,0.533879,-0.332314,0.265557,-0.187679,-0.046176,0.890306,0.571335
U0003,0.619637,0.190055,0.407952,1.127847,1.793985,0.636406,0.586721,0.809601,-0.849986,0.394444,...,0.249053,0.284071,-0.528648,-0.285156,-0.338960,0.151429,-1.253169,-0.191231,0.323353,2.415320
U0004,0.469020,0.372101,-0.464408,0.598238,0.555990,-0.014999,-0.636849,0.371580,0.012240,0.400223,...,0.439999,-0.357597,-0.580622,-0.294042,-0.826767,0.056605,-0.276070,-0.122519,0.890333,0.794065
U0005,0.345642,0.686870,0.461311,0.555779,0.004166,0.455593,0.915826,0.800484,2.404162,0.312264,...,0.368950,0.309292,0.415049,1.664246,0.296885,0.284136,-0.146680,0.549584,0.447987,0.598749


Objective 2

In [ ]:
def recommend_movies(user_id, n=5):

    user_ratings = predicted_ratings.loc[user_id]

    already_watched = df[df.customer_id == user_id]["movie_name"]

    recommendations = user_ratings.drop(already_watched)

    top_movies = recommendations.sort_values(ascending=False).head(n)

    return top_movies

recommend_movies("U0001")

,U0001
movie_name,
Ex Machina,1.073744
Dead Poets Society,0.994899
Finding Nemo,0.986564
Monty Python,0.986007
Gladiator,0.829847


In [ ]:
genre_movies = df.groupby(['genre','movie_name'])['rating'].mean().reset_index()

best_movies_per_genre = genre_movies.sort_values(
    ['genre','rating'], ascending=False
).drop_duplicates('genre')

print(best_movies_per_genre)

          genre         movie_name    rating
96     Thriller              Se7en  3.967213
77       Sci-Fi       Annihilation  3.192090
69      Romance         La La Land  3.209877
62       Horror       The Exorcist  3.141104
47      Fantasy       Harry Potter  3.658031
36        Drama       12 Angry Men  3.965517
34  Documentary       Planet Earth  3.187500
29       Comedy       The Hangover  3.239766
17    Animation          Toy Story  4.296089
13       Action  Top Gun: Maverick  3.144444


Objective 3

In [ ]:
genre_rating = df.groupby("genre")["rating"].mean()

best_genre = genre_rating.idxmax()
worst_genre = genre_rating.idxmin()

print("Best Rated Genre:", best_genre)
print("Worst Rated Genre:", worst_genre)

Best Rated Genre: Animation
Worst Rated Genre: Horror


In [ ]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(matrix, reconstructed_matrix))

print("RMSE of SVD Model:", rmse)

RMSE of SVD Model: 0.9828167593411593
